# 使用 LoRA 微调问答对话大模型

这个 notebook 演示如何在笔记本电脑上用 LoRA 微调一个小型中文问答对话模型。默认模型为 `Qwen/Qwen2.5-0.5B-Instruct`，适合课程实验和小数据跑通流程。

建议硬件：
- 有 NVIDIA GPU：推荐，4GB 显存可尝试小 batch 和短序列。
- 只有 CPU：可以跑通极小数据实验，但训练会很慢。

训练产物：只保存 LoRA adapter，不保存完整基座模型。

## 1. 安装依赖

如果你当前环境还没有 PyTorch/Transformers/PEFT，先运行下面单元。安装后建议重启 Jupyter kernel。

In [ ]:
# 如已安装可跳过。
# CPU 版本 PyTorch：
%pip install -U torch transformers peft accelerate sentencepiece

# NVIDIA GPU 用户通常可直接先尝试上面命令；如需指定 CUDA 版本，请参考 PyTorch 官网安装命令。
# 本 notebook 不强制安装 bitsandbytes，因为 Windows 笔记本上 bitsandbytes 兼容性经常成为额外问题。

## 2. 导入库与基础配置

In [2]:
import math
import os
from dataclasses import dataclass

import torch
from torch.utils.data import Dataset, DataLoader
from transformers import AutoModelForCausalLM, AutoTokenizer, get_linear_schedule_with_warmup
from peft import LoraConfig, TaskType, get_peft_model, PeftModel

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)
if device == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
    print("显存 GB:", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))

d:\Anaconda\envs\pytorch\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


device: cuda
GPU: NVIDIA GeForce RTX 4060 Laptop GPU
显存 GB: 8.0


## 3. 准备问答对话数据

真实项目中建议使用 JSONL，每行一个样本，例如：

```json
{"instruction": "什么是反向传播？", "answer": "反向传播是一种..."}
```

下面先用少量课程问答样例跑通流程。你可以把 `raw_data` 替换成自己的数据。

In [3]:
raw_data = [
    {
        "instruction": "什么是过拟合？",
        "answer": "过拟合是指模型在训练集上表现很好，但在新数据或测试集上表现较差的现象。常见原因是模型复杂度过高、训练数据太少或训练时间过长。常见缓解方法包括增加数据、数据增强、正则化、Dropout 和早停。",
    },
    {
        "instruction": "请解释梯度下降的基本思想。",
        "answer": "梯度下降是一种优化算法。它通过计算损失函数对参数的梯度，沿着损失下降最快的方向逐步更新参数，从而让模型预测误差变小。学习率控制每一步更新的幅度。",
    },
    {
        "instruction": "CNN 为什么适合处理图像？",
        "answer": "CNN 适合处理图像，是因为卷积核可以提取局部空间特征，参数共享能减少参数量，池化操作可以增强一定的平移不变性。多层卷积还能逐步学习从边缘、纹理到高级语义的特征。",
    },
    {
        "instruction": "什么是学习率？学习率过大或过小会怎样？",
        "answer": "学习率决定模型参数每次更新的步长。学习率过大可能导致训练震荡甚至发散；学习率过小会让收敛速度很慢，也可能停在效果不佳的位置。实际训练中常使用学习率调度器动态调整。",
    },
    {
        "instruction": "简述 Transformer 中自注意力机制的作用。",
        "answer": "自注意力机制让序列中每个位置都能根据相关性关注其他位置的信息，从而捕捉长距离依赖。它通过 Query、Key、Value 计算注意力权重，再对 Value 加权求和得到上下文表示。",
    },
    {
        "instruction": "为什么微调大模型时常用 LoRA？",
        "answer": "LoRA 会冻结原始大模型参数，只在部分线性层旁边训练低秩矩阵。这样可训练参数显著减少，显存和存储开销更小，训练速度也更适合普通实验环境。",
    },
]

len(raw_data), raw_data[0]

(6,
 {'instruction': '什么是过拟合？',
  'answer': '过拟合是指模型在训练集上表现很好，但在新数据或测试集上表现较差的现象。常见原因是模型复杂度过高、训练数据太少或训练时间过长。常见缓解方法包括增加数据、数据增强、正则化、Dropout 和早停。'})

如果你有自己的 JSONL 文件，可以使用下面的函数读取。字段名保持为 `instruction` 和 `answer`。

In [ ]:
import json

def load_jsonl(path):
    data = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                data.append(json.loads(line))
    return data

# 示例：取消注释并替换路径
# raw_data = load_jsonl("my_qa_data.jsonl")
# print(len(raw_data))

## 4. 加载基座模型和 tokenizer

首次运行会从 Hugging Face 下载模型，需要网络。下载完成后会缓存在本机。

In [5]:
MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

torch_dtype = torch.float32
if device == "cuda":
    torch_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch_dtype,
    trust_remote_code=True,
)
model.to(device)
print("loaded", MODEL_ID)

d:\Anaconda\envs\pytorch\lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\ASUS\.cache\huggingface\hub\models--Qwen--Qwen2.5-0.5B-Instruct. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
`torch_dtype` is deprecated! Use `dtype` instead!
Xet Storage is enabled for this repo, but the 'hf_xet' package is no

loaded Qwen/Qwen2.5-0.5B-Instruct


## 5. 推理基座模型，观察微调前效果

In [6]:
def chat_generate(model, tokenizer, question, max_new_tokens=160):
    model.eval()
    messages = [
        {"role": "system", "content": "你是一个耐心、准确的深度学习课程助教。"},
        {"role": "user", "content": question},
    ]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )
    new_tokens = output_ids[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True)

print(chat_generate(model, tokenizer, "什么是 LoRA？为什么它适合在笔记本上微调大模型？"))

LoRA（Low-Rank Approximation）是一种在大规模语言模型中实现低秩近似的方法，主要用于在大规模数据集上训练模型，以提高其泛化能力和效率。下面是一些关于LoRA和如何在笔记本上进行微调的大模型的信息：

### 1. LoRA简介

LoRA最初是用于处理大规模文本数据的，特别是在文本生成任务上的。通过使用一个较小的矩阵来代替传统的大型矩阵，LoRA能够显著减少计算量，并且在一些情况下可以保持较好的性能。

### 2. 在大规模语言模型中的应用

在大规模语言模型中，LoRA被用来优化预训练模型的参数，使其更加高效。例如，在BERT等大型预训练模型上，通过LoRA可以在不牺牲模型


## 6. 构造监督微调数据集

关键点：只对 assistant 的回答计算 loss，用户问题和系统提示不参与 loss。这样模型学习的是如何回答，而不是复读 prompt。

In [7]:
SYSTEM_PROMPT = "你是一个耐心、准确的深度学习课程助教。回答要简洁、清楚，适合初学者。"
MAX_LENGTH = 512

class ChatQADataset(Dataset):
    def __init__(self, rows, tokenizer, max_length=512):
        self.rows = rows
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, idx):
        row = self.rows[idx]
        prompt_messages = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": row["instruction"]},
        ]
        prompt = self.tokenizer.apply_chat_template(
            prompt_messages,
            tokenize=False,
            add_generation_prompt=True,
        )
        answer = row["answer"] + self.tokenizer.eos_token
        full_text = prompt + answer

        full = self.tokenizer(full_text, truncation=True, max_length=self.max_length)
        prompt_ids = self.tokenizer(prompt, truncation=True, max_length=self.max_length)["input_ids"]

        input_ids = full["input_ids"]
        attention_mask = full["attention_mask"]
        labels = input_ids.copy()
        prompt_len = min(len(prompt_ids), len(labels))
        labels[:prompt_len] = [-100] * prompt_len

        return {
            "input_ids": torch.tensor(input_ids, dtype=torch.long),
            "attention_mask": torch.tensor(attention_mask, dtype=torch.long),
            "labels": torch.tensor(labels, dtype=torch.long),
        }

@dataclass
class DataCollatorForCausalLM:
    tokenizer: object

    def __call__(self, features):
        input_ids = [f["input_ids"] for f in features]
        attention_mask = [f["attention_mask"] for f in features]
        labels = [f["labels"] for f in features]

        batch_input_ids = torch.nn.utils.rnn.pad_sequence(
            input_ids, batch_first=True, padding_value=self.tokenizer.pad_token_id
        )
        batch_attention_mask = torch.nn.utils.rnn.pad_sequence(
            attention_mask, batch_first=True, padding_value=0
        )
        batch_labels = torch.nn.utils.rnn.pad_sequence(
            labels, batch_first=True, padding_value=-100
        )
        return {
            "input_ids": batch_input_ids,
            "attention_mask": batch_attention_mask,
            "labels": batch_labels,
        }

dataset = ChatQADataset(raw_data, tokenizer, max_length=MAX_LENGTH)
collator = DataCollatorForCausalLM(tokenizer)
sample = dataset[0]
print({k: v.shape for k, v in sample.items()})

{'input_ids': torch.Size([100]), 'attention_mask': torch.Size([100]), 'labels': torch.Size([100])}


## 7. 配置 LoRA

LoRA 只训练少量低秩矩阵。下面的 `target_modules` 适用于 Qwen/Llama 风格模型。

In [8]:
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

if hasattr(model, "gradient_checkpointing_enable"):
    model.gradient_checkpointing_enable()
if hasattr(model, "enable_input_require_grads"):
    model.enable_input_require_grads()
model.config.use_cache = False

trainable params: 4,399,104 || all params: 498,431,872 || trainable%: 0.8826


## 8. 训练

这是一个教学版训练循环。笔记本上建议先用很小的数据和 1 到 3 个 epoch 跑通，再逐步增加数据量。

In [10]:
BATCH_SIZE = 1
GRAD_ACCUM_STEPS = 4
EPOCHS = 30
LEARNING_RATE = 2e-4
WARMUP_RATIO = 0.05

train_loader = DataLoader(
    dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=collator,
)

optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE)
num_update_steps = math.ceil(len(train_loader) / GRAD_ACCUM_STEPS) * EPOCHS
num_warmup_steps = max(1, int(num_update_steps * WARMUP_RATIO))
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=num_warmup_steps,
    num_training_steps=num_update_steps,
)

model.train()
global_step = 0
optimizer.zero_grad(set_to_none=True)

for epoch in range(EPOCHS):
    total_loss = 0.0
    for step, batch in enumerate(train_loader, start=1):
        batch = {k: v.to(device) for k, v in batch.items()}
        outputs = model(**batch)
        loss = outputs.loss / GRAD_ACCUM_STEPS
        loss.backward()
        total_loss += loss.item() * GRAD_ACCUM_STEPS

        if step % GRAD_ACCUM_STEPS == 0 or step == len(train_loader):
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad(set_to_none=True)
            global_step += 1

    avg_loss = total_loss / len(train_loader)
    print(f"epoch {epoch + 1}/{EPOCHS} - loss: {avg_loss:.4f}")

epoch 1/30 - loss: 2.0261
epoch 2/30 - loss: 1.8616
epoch 3/30 - loss: 1.3650
epoch 4/30 - loss: 0.9244
epoch 5/30 - loss: 0.6097
epoch 6/30 - loss: 0.3793
epoch 7/30 - loss: 0.2219
epoch 8/30 - loss: 0.1103
epoch 9/30 - loss: 0.0578
epoch 10/30 - loss: 0.0270
epoch 11/30 - loss: 0.0132
epoch 12/30 - loss: 0.0085
epoch 13/30 - loss: 0.0056
epoch 14/30 - loss: 0.0039
epoch 15/30 - loss: 0.0030
epoch 16/30 - loss: 0.0025
epoch 17/30 - loss: 0.0020
epoch 18/30 - loss: 0.0018
epoch 19/30 - loss: 0.0016
epoch 20/30 - loss: 0.0014
epoch 21/30 - loss: 0.0013
epoch 22/30 - loss: 0.0012
epoch 23/30 - loss: 0.0012
epoch 24/30 - loss: 0.0011
epoch 25/30 - loss: 0.0010
epoch 26/30 - loss: 0.0010
epoch 27/30 - loss: 0.0010
epoch 28/30 - loss: 0.0010
epoch 29/30 - loss: 0.0010
epoch 30/30 - loss: 0.0010


## 9. 保存 LoRA adapter

In [11]:
ADAPTER_DIR = "./qwen_lora_qa_adapter"
model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)
print("saved adapter to", ADAPTER_DIR)

'(MaxRetryError("HTTPSConnectionPool(host='huggingface.co', port=443): Max retries exceeded with url: /Qwen/Qwen2.5-0.5B-Instruct/resolve/main/config.json (Caused by ConnectTimeoutError(<HTTPSConnection(host='huggingface.co', port=443) at 0x157a253ca00>, 'Connection to huggingface.co timed out. (connect timeout=10)'))"), '(Request ID: 5affba6b-73ff-47d5-aec7-ae40d927885e)')' thrown while requesting HEAD https://huggingface.co/Qwen/Qwen2.5-0.5B-Instruct/resolve/main/config.json
Retrying in 1s [Retry 1/5].
'(MaxRetryError("HTTPSConnectionPool(host='huggingface.co', port=443): Max retries exceeded with url: /Qwen/Qwen2.5-0.5B-Instruct/resolve/main/config.json (Caused by ConnectTimeoutError(<HTTPSConnection(host='huggingface.co', port=443) at 0x157a253c7f0>, 'Connection to huggingface.co timed out. (connect timeout=10)'))"), '(Request ID: 2d4746e3-5d72-4c3d-a5fa-040f20b6b55c)')' thrown while requesting HEAD https://huggingface.co/Qwen/Qwen2.5-0.5B-Instruct/resolve/main/config.json
Retrying

saved adapter to ./qwen_lora_qa_adapter


## 10. 微调后测试

In [14]:
print(chat_generate(model, tokenizer, "为什么微调大模型时常用 LoRA？"))

LoRA 会冻结原始大模型参数，只在部分线性层旁边训练低秩矩阵。这样可训练参数显著减少，显存和存储开销更小，训练速度也更适合普通实验环境。多线性层旁边训练低秩矩阵还可以让训练效果优于冻结参数。


## 11. 单独加载 adapter 推理

以后不需要重新训练，只要加载基座模型和 adapter 即可。

In [15]:
# 释放当前模型后再演示加载，可按需运行。
# del model
# if device == "cuda":
#     torch.cuda.empty_cache()

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch_dtype,
    trust_remote_code=True,
).to(device)
loaded_model = PeftModel.from_pretrained(base_model, ADAPTER_DIR).to(device)

print(chat_generate(loaded_model, tokenizer, "请用简单的话解释什么是过拟合。"))

过拟合是指模型在训练集上表现很好，但在新数据或测试集上表现较差的现象。常见原因是模型复杂度过高、训练数据太少或训练时间过长。常见缓解方法包括增加数据、数据增强、正则化、Dropout 和早停。


## 12. 调参建议

- 显存不够：减小 `MAX_LENGTH`、保持 `BATCH_SIZE=1`、增大 `GRAD_ACCUM_STEPS`。
- 效果不明显：增加高质量问答数据，训练 2 到 5 个 epoch，避免只堆数量。
- 回答变啰嗦：在数据答案中统一风格，并在 `SYSTEM_PROMPT` 中明确回答长度。
- 训练发散：把 `LEARNING_RATE` 从 `2e-4` 降到 `1e-4` 或 `5e-5`。
- 想更省显存：可在 Linux 或兼容环境尝试 bitsandbytes 4-bit QLoRA。